# 🔍 AI Smart Resale Inspector — YOLOv8 Damage Detection Training

Train a real damage detection model (scratch / crack / dent / stain) on a free Google Colab T4 GPU.  
After training you download one file (`yolov8n_damage.pt`) and drop it into your project.

**Estimated time:** ~20–40 minutes on a free Colab T4 GPU

---

### Before you start
1. Make sure the runtime is set to **GPU**: `Runtime → Change runtime type → T4 GPU`
2. Run cells **in order**, top to bottom (Shift+Enter or click ▶)

## Cell 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
    print('✅ GPU is available!')
else:
    print('❌ No GPU detected.')
    print('   Go to Runtime → Change runtime type → T4 GPU, then reconnect.')

## Cell 2 — Install dependencies

In [ ]:
!pip install ultralytics roboflow --quiet
import ultralytics
print(f'ultralytics {ultralytics.__version__} installed ✅')

## Cell 3 — Mount Google Drive (to save weights permanently)

When this cell runs you will be asked to allow Colab access to your Drive.  
Your trained weights will be saved at `My Drive/resale_inspector/yolov8n_damage.pt`  
so they survive if the Colab session disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/resale_inspector'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'Drive mounted. Weights will be saved to: {DRIVE_SAVE_DIR}')

## Cell 4 — Download Dataset from Roboflow

**How to get your API key and dataset info:**

1. Go to [https://universe.roboflow.com](https://universe.roboflow.com)
2. Search: **`phone damage detection`** or **`surface defect yolov8`**
3. Open a dataset with 500+ images → click **"Download Dataset"**
4. Choose format: **YOLOv8** → **"Show download code"** → copy the 3 values below
5. Sign in / create a free Roboflow account if prompted

**Paste your values in the three fields below, then run the cell.**

> 💡 Recommended datasets (search these on universe.roboflow.com):
> - `phone-damage-detection` by various authors
> - `mobile-phone-defect-detection`
> - `surface-defect-detection-v2`

In [ ]:
# ── FILL IN THESE THREE VALUES ────────────────────────────────────────────────
ROBOFLOW_API_KEY   = "PASTE_YOUR_API_KEY_HERE"      # from Roboflow → Settings → API
ROBOFLOW_WORKSPACE = "PASTE_WORKSPACE_NAME_HERE"    # e.g. "my-workspace"
ROBOFLOW_PROJECT   = "PASTE_PROJECT_NAME_HERE"      # e.g. "phone-damage"
ROBOFLOW_VERSION   = 1                              # dataset version number
# ─────────────────────────────────────────────────────────────────────────────

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("yolov8", location="/content/dataset")

print(f'\nDataset downloaded to: {dataset.location}')
print(f'Classes: {dataset.classes}')

# Save the dataset.yaml path for the training cell
import glob
yaml_files = glob.glob('/content/dataset/**/*.yaml', recursive=True)
DATA_YAML = yaml_files[0] if yaml_files else '/content/dataset/data.yaml'
print(f'YAML config: {DATA_YAML}')

## Cell 4b — Alternative: Upload your own dataset (skip if you used Cell 4)

If you already have a YOLOv8 dataset zip on your computer, run this cell instead of Cell 4.

In [ ]:
# ── OPTION: upload a zip file from your computer ──────────────────────────────
# Only run this if you SKIPPED Cell 4.

from google.colab import files
import zipfile, os, glob

print('Select your dataset .zip file...')
uploaded = files.upload()  # opens file picker

zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/dataset')

yaml_files = glob.glob('/content/dataset/**/*.yaml', recursive=True)
DATA_YAML = yaml_files[0] if yaml_files else '/content/dataset/data.yaml'
print(f'Extracted to /content/dataset')
print(f'YAML config: {DATA_YAML}')

## Cell 5 — Preview the dataset

In [ ]:
import yaml, os
from pathlib import Path

with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

print('Dataset config:')
print(f'  Classes ({cfg["nc"]}): {cfg["names"]}')

# Count images
dataset_root = Path(DATA_YAML).parent
for split in ['train', 'val', 'test']:
    img_dir = dataset_root / 'images' / split
    if img_dir.exists():
        n = len(list(img_dir.glob('*.*')))
        print(f'  {split}: {n} images')

## Cell 6 — Train the model

This trains YOLOv8n for 100 epochs on your dataset.  
**~20–40 minutes on a free T4 GPU.**  

Watch the output — `mAP50` tells you the accuracy:  
- `0.50` = 50% accurate (fair baseline)
- `0.70` = 70% accurate (good)
- `0.85+` = excellent

In [ ]:
from ultralytics import YOLO

# Load the smallest YOLOv8 — fastest to train, still very accurate
model = YOLO('yolov8n.pt')

results = model.train(
    data      = DATA_YAML,
    epochs    = 100,
    imgsz     = 640,
    batch     = 16,
    device    = 0,        # GPU 0 (T4)
    patience  = 20,       # stop early if val loss stalls
    project   = '/content/runs',
    name      = 'damage',
    save      = True,
    plots     = True,
    val       = True,
    cache     = True,     # cache images in RAM → faster training
    augment   = True,     # random flip, crop, colour jitter
)

# Final metrics
metrics = results.results_dict
print()
print('=' * 50)
print(f"  mAP50:    {metrics.get('metrics/mAP50(B)',    '?')}")
print(f"  mAP50-95: {metrics.get('metrics/mAP50-95(B)', '?')}")
print('=' * 50)

## Cell 7 — Export to ONNX and save to Drive

In [ ]:
import shutil, glob
from ultralytics import YOLO

# Find best.pt from the training run
best_pt = '/content/runs/damage/weights/best.pt'
print(f'Loading best weights from: {best_pt}')

best_model = YOLO(best_pt)

# ── Export to ONNX ────────────────────────────────────────────────────────────
print('Exporting to ONNX...')
best_model.export(format='onnx', imgsz=640, simplify=True, dynamic=False)

# ── Copy to Google Drive ──────────────────────────────────────────────────────
pt_dest   = f'{DRIVE_SAVE_DIR}/yolov8n_damage.pt'
onnx_src  = best_pt.replace('.pt', '.onnx')
onnx_dest = f'{DRIVE_SAVE_DIR}/yolov8n_damage.onnx'

shutil.copy(best_pt,  pt_dest)
print(f'Saved PyTorch model → {pt_dest}')

if __import__('os').path.exists(onnx_src):
    shutil.copy(onnx_src, onnx_dest)
    print(f'Saved ONNX model    → {onnx_dest}')

# Also copy training plots
for plot in glob.glob('/content/runs/damage/*.png'):
    shutil.copy(plot, DRIVE_SAVE_DIR)

print()
print('✅ Weights saved to Google Drive!')
print(f'   Open Google Drive and download: yolov8n_damage.pt')

## Cell 8 — Download weights directly to your computer

This triggers a browser download for the `.pt` file.  
You can skip this if you already saved to Drive in Cell 7.

In [ ]:
from google.colab import files

print('Downloading yolov8n_damage.pt to your computer...')
files.download('/content/runs/damage/weights/best.pt')

print()
print('After download, rename the file to: yolov8n_damage.pt')
print('Then place it in your project at:')
print('  ml-models/weights/yolov8n_damage.pt')

## Cell 9 — View training results

Shows the accuracy curves. `mAP50` going up means the model is learning.

In [ ]:
from IPython.display import Image, display
import os

plots = [
    ('/content/runs/damage/results.png',          'Training curves'),
    ('/content/runs/damage/confusion_matrix.png', 'Confusion matrix'),
    ('/content/runs/damage/val_batch0_pred.jpg',  'Validation predictions'),
]

for path, title in plots:
    if os.path.exists(path):
        print(f'\n── {title} ──')
        display(Image(filename=path, width=800))

## ✅ Next steps

1. **Download `yolov8n_damage.pt`** (Cell 8 above, or from Google Drive)
2. **Place it in your project**: `ml-models/weights/yolov8n_damage.pt`
3. **Restart the Python ML service**:
   ```powershell
   python ml-models/service.py
   ```
4. The service automatically detects and loads `yolov8n_damage.pt` — no code changes needed!

---

### Improving accuracy further

| Action | Expected gain |
|---|---|
| Get more labeled images (500 → 2000) | +10–15 mAP |
| Use `yolov8s.pt` instead of `yolov8n.pt` | +3–5 mAP |
| Label your own phones/laptops | Best for your specific use case |
| Train for 200 epochs instead of 100 | +2–4 mAP |